# Data Preprocessing — Master Pipeline Summary

এই নোটবুক **শুধু preprocessing/audit stage** এর জন্য। ৬৩টা `.las` ফাইলের প্রতিটার উপর
load → voxel downsample → feature/normal computation → nn-distance stats →
**৫টা chunk-sampling variant-এর coverage simulation (RandomGrid, SequentialGrid,
SlidingWindow, CoverageBased, FPSCenters)** সব চালিয়ে একটা row বানায়, তারপর সব row
একসাথে `master_pipeline_summary.csv` এ save করে।

Training/segmentation/volume কিছুই এখানে হয় না — এটা independent audit script।
Run top → bottom, শেষে CSV তৈরি হবে।

In [28]:
# ── Imports ──────────────────────────────────────────────────────────────────
import os, glob, logging
import numpy as np
import pandas as pd
import laspy
import open3d as o3d
from scipy.spatial import cKDTree
from tqdm.auto import tqdm

logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger(__name__)

In [29]:
# ── Config — শুধু path আর params, বাকি সব নিচে auto চলবে ──────────────────────
CONFIG = {
    "data_dir"       : "data/train_v2/",   # ৬৩টা labelled .las ফাইল যেখানে আছে
    "output_csv"     : "outputs/master_pipeline_summary.csv",

    # ── Stage: voxel downsample (train notebook-এর সাথে identical param) ──────
    "voxel_size"     : 0.01,     # 1cm voxel

    # ── Stage: normal estimation (train notebook-এর সাথে identical param) ─────
    "normal_radius"  : 0.05,
    "normal_max_nn"  : 30,

    # ── Stage: nn-distance stat computation ────────────────────────────────────
    "nn_k"           : 1,        # 1 = nearest neighbor distance (excluding self)
    "nn_max_points"  : 200_000,  # speed guard: বড় ফাইলে random subsample করে nn distance বের করা হবে

    # ── Stage: chunk-sampling coverage simulation (5 variants, train notebook-এর
    #    সাথে identical params — শুধু কভারেজ % মাপার জন্য, augmentation ছাড়া) ──
    "chunk_num_points"   : 16384,   # == CONFIG["num_points"] in the main notebook
    "chunks_per_cloud"   : 100,
    "chunk_grid"         : 6,       # V1/V2/V4/V5 এর 6x6 grid
    "window_frac"        : 1/6,     # V3 sliding window
    "window_overlap"     : 0.5,     # V3 sliding window
    "coverage_seed"      : 42,      # deterministic simulation, প্রতিবার একই ফলাফল

    "num_classes"    : 2,        # 0 = environment, 1 = wood powder
    "target_class"   : 1,
}

os.makedirs(os.path.dirname(CONFIG["output_csv"]), exist_ok=True)

In [30]:
# ── Stage 1: Load (identical logic to load_pointcloud in the main notebook) ──
def load_pointcloud(path):
    '''Read a .las/.laz file. Returns (points[N,3], labels[N] or None, meta dict).'''
    las = laspy.read(path)
    pts_raw = np.column_stack([np.asarray(las.x),
                               np.asarray(las.y),
                               np.asarray(las.z)]).astype(np.float64)

    labels = None
    label_field = None
    for key in ("classification", "label", "labels", "class"):
        if key in las.point_format.dimension_names:
            labels = np.asarray(getattr(las, key), dtype=np.int64)
            label_field = key
            break

    total_points = pts_raw.shape[0]

    # non-finite check (POINT 1 logic)
    finite = np.isfinite(pts_raw).all(axis=1)
    nan_count = int(np.isnan(pts_raw).any(axis=1).sum())
    inf_count = int(np.isinf(pts_raw).any(axis=1).sum())

    pts = pts_raw[finite]
    lbl = labels[finite] if labels is not None else None

    meta = {
        "xyz_exists"             : True,
        "classification_exists"  : labels is not None,
        "label_field"             : label_field,
        "total_points"            : total_points,
        "nan_count"               : nan_count,
        "inf_count"               : inf_count,
        "non_finite_dropped"      : int(total_points - len(pts)),
    }
    return pts, lbl, meta


def list_files(folder):
    files = []
    for ext in (".las", ".laz"):
        files += glob.glob(os.path.join(folder, "*" + ext))
    return sorted(files)

In [31]:
# ── Stage 2: duplicate-point check (raw points, before any downsampling) ─────
def duplicate_stats(pts):
    n_total = len(pts)
    if n_total == 0:
        return {"unique_points": 0, "duplicate_points": 0, "duplicate_percentage": 0.0}
    unique_pts = np.unique(pts, axis=0)
    n_unique = len(unique_pts)
    n_dup = n_total - n_unique
    return {
        "unique_points"        : int(n_unique),
        "duplicate_points"     : int(n_dup),
        "duplicate_percentage" : round(100.0 * n_dup / n_total, 4),
    }

In [32]:
# ── Stage 3: voxel downsample (identical logic to _voxel_keep) ───────────────
def voxel_keep(pts, voxel):
    '''Returns indices of the kept (centroid-nearest) point per voxel.'''
    vox = np.floor(pts / voxel).astype(np.int64)
    _, inv, cnt = np.unique(vox, axis=0, return_inverse=True, return_counts=True)
    sums = np.zeros((cnt.size, 3), np.float64)
    np.add.at(sums, inv, pts)
    d2 = ((pts - (sums / cnt[:, None])[inv]) ** 2).sum(1)
    order = np.lexsort((d2, inv))
    first = np.concatenate([[0], np.cumsum(cnt)[:-1]])
    return np.sort(order[first])


def voxel_stage_stats(pts, voxel_size):
    if len(pts) == 0:
        return {"points_after_voxel": 0, "voxel_reduction_pct": 0.0}, np.array([], dtype=np.int64)
    keep_idx = voxel_keep(pts, voxel_size)
    n_after = len(keep_idx)
    reduction_pct = round(100.0 * (1 - n_after / len(pts)), 4)
    return {
        "points_after_voxel"  : int(n_after),
        "voxel_reduction_pct" : reduction_pct,
    }, keep_idx

In [33]:
# ── Stage 4: normals + height (identical logic to make_features, but only
#    summary stats are kept — full per-point feature array isn't needed here) ─
def normal_and_height_stats(points, normal_radius, normal_max_nn):
    center = points.mean(axis=0, keepdims=True)
    scale  = max(np.linalg.norm(points - center, axis=1).max(), 1e-9)

    z = points[:, 2]
    height = (z - z.min()) / max(z.max() - z.min(), 1e-6)

    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points.astype(np.float64))
    pcd.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(
        radius=normal_radius, max_nn=normal_max_nn))
    pcd.orient_normals_to_align_with_direction([0., 0., 1.])
    normals = np.asarray(pcd.normals)

    return {
        "centroid_x"      : round(float(center[0, 0]), 6),
        "centroid_y"      : round(float(center[0, 1]), 6),
        "centroid_z"      : round(float(center[0, 2]), 6),
        "scale_norm"      : round(float(scale), 6),
        "x_std"           : round(float(points[:, 0].std()), 6),
        "y_std"           : round(float(points[:, 1].std()), 6),
        "z_std"           : round(float(points[:, 2].std()), 6),
        "height_min"      : round(float(height.min()), 6),
        "height_max"      : round(float(height.max()), 6),
        "mean_normal_z"   : round(float(normals[:, 2].mean()), 6) if len(normals) else float("nan"),
    }

In [34]:
# ── Stage 5: nearest-neighbor distance stats (scan density check) ────────────
# বড় ফাইলে (>nn_max_points) speed-এর জন্য random subsample করে nn distance বের করা হয়।
def nn_distance_stats(pts, k=1, max_points=200_000, seed=0):
    n = len(pts)
    if n < 3:
        return {k: 0.0 for k in
                ["max_nn_distance", "mean_nn_distance", "median_nn_distance",
                 "std_nn_distance", "p5_nn_distance", "p95_nn_distance"]}

    if n > max_points:
        rng = np.random.RandomState(seed)
        sub_idx = rng.choice(n, max_points, replace=False)
        query_pts = pts[sub_idx]
    else:
        query_pts = pts

    tree = cKDTree(pts)
    d, _ = tree.query(query_pts, k=k + 1)   # col 0 = self (dist 0)
    nn_d = d[:, 1:].ravel()

    return {
        "max_nn_distance"    : round(float(nn_d.max()), 6),
        "mean_nn_distance"   : round(float(nn_d.mean()), 6),
        "median_nn_distance" : round(float(np.median(nn_d)), 6),
        "std_nn_distance"    : round(float(nn_d.std()), 6),
        "p5_nn_distance"     : round(float(np.percentile(nn_d, 5)), 6),
        "p95_nn_distance"    : round(float(np.percentile(nn_d, 95)), 6),
    }

In [35]:
# ── Stage 6: label distribution (raw + post-voxel, both reported) ────────────
def label_stats(lbl, target_class, prefix=""):
    if lbl is None or len(lbl) == 0:
        return {f"{prefix}target_points": 0, f"{prefix}non_target_points": 0,
                f"{prefix}target_ratio_pct": 0.0, "classifications_present": []}
    n_total = len(lbl)
    n_target = int((lbl == target_class).sum())
    n_other  = n_total - n_target
    present  = sorted(np.unique(lbl).tolist())
    return {
        f"{prefix}target_points"     : n_target,
        f"{prefix}non_target_points" : n_other,
        f"{prefix}target_ratio_pct"  : round(100.0 * n_target / n_total, 4),
        "classifications_present"    : present,
    }

In [36]:
# ── Stage 7: chunk-sampling coverage simulation — 5 variants ─────────────────
# main training notebook-এর ৫টা PointCloudDataset variant-এর chunk-selection
# logic হুবহু reproduce করা হয়েছে (augmentation বাদে), শুধু coverage % মাপার জন্য।
# xy = post-voxel points-এর প্রথম দুই কলাম (x, y) — grid/window boundary একই থাকে
# normalized বা raw xyz দিয়ে, কারণ শুধু relative partitioning matter করে।

def _pool_indices(xy, cx, cy, half):
    return np.flatnonzero((np.abs(xy[:, 0] - cx) < half) &
                          (np.abs(xy[:, 1] - cy) < half))


def _knn_fallback(xy, seed_idx, N):
    dist = ((xy - xy[seed_idx]) ** 2).sum(1)
    return np.argpartition(dist, N - 1)[:N]


def coverage_v1_random_grid(xy, N, chunks, grid, seed):
    rng = np.random.RandomState(seed)
    n = len(xy)
    if n <= N:
        return 100.0
    covered = np.zeros(n, bool)
    x0, x1 = xy[:, 0].min(), xy[:, 0].max()
    y0, y1 = xy[:, 1].min(), xy[:, 1].max()
    col = max(x1 - x0, y1 - y0) / grid + 1e-9
    for _ in range(chunks):
        cx, cy = rng.uniform(x0, x1), rng.uniform(y0, y1)
        pool = _pool_indices(xy, cx, cy, col)
        chosen = (_knn_fallback(xy, rng.randint(n), N) if len(pool) < N // 4
                  else rng.choice(pool, N, replace=len(pool) < N))
        covered[chosen] = True
    return round(100.0 * covered.sum() / n, 3)


def coverage_v2_sequential_grid(xy, N, chunks, grid, seed):
    rng = np.random.RandomState(seed)
    n = len(xy)
    if n <= N:
        return 100.0
    covered = np.zeros(n, bool)
    x0, x1 = xy[:, 0].min(), xy[:, 0].max()
    y0, y1 = xy[:, 1].min(), xy[:, 1].max()
    col = max(x1 - x0, y1 - y0) / grid + 1e-9
    for cell_idx in range(chunks):
        gx = (cell_idx % (grid * grid)) % grid
        gy = (cell_idx % (grid * grid)) // grid
        cx = x0 + (gx + 0.5) * (x1 - x0) / grid
        cy = y0 + (gy + 0.5) * (y1 - y0) / grid
        pool = _pool_indices(xy, cx, cy, col)
        chosen = (_knn_fallback(xy, rng.randint(n), N) if len(pool) < N // 4
                  else rng.choice(pool, N, replace=len(pool) < N))
        covered[chosen] = True
    return round(100.0 * covered.sum() / n, 3)


def coverage_v3_sliding_window(xy, N, chunks, window_frac, overlap, seed):
    rng = np.random.RandomState(seed)
    n = len(xy)
    if n <= N:
        return 100.0
    covered = np.zeros(n, bool)
    x0, x1 = xy[:, 0].min(), xy[:, 0].max()
    y0, y1 = xy[:, 1].min(), xy[:, 1].max()
    win = max(x1 - x0, y1 - y0) * window_frac + 1e-9
    stride = win * (1 - overlap)
    n_steps_x = max(int((x1 - x0) / stride), 1)
    n_steps_y = max(int((y1 - y0) / stride), 1)
    for step_idx in range(chunks):
        sx = step_idx % n_steps_x
        sy = (step_idx // n_steps_x) % n_steps_y
        cx = x0 + win / 2 + sx * stride
        cy = y0 + win / 2 + sy * stride
        pool = _pool_indices(xy, cx, cy, win / 2)
        chosen = (_knn_fallback(xy, rng.randint(n), N) if len(pool) < N // 4
                  else rng.choice(pool, N, replace=len(pool) < N))
        covered[chosen] = True
    return round(100.0 * covered.sum() / n, 3)


def coverage_v4_coverage_based(xy, N, chunks, grid, seed):
    rng = np.random.RandomState(seed)
    n = len(xy)
    if n <= N:
        return 100.0
    visits = np.zeros(n, np.int32)
    covered = np.zeros(n, bool)
    x0, x1 = xy[:, 0].min(), xy[:, 0].max()
    y0, y1 = xy[:, 1].min(), xy[:, 1].max()
    col = max(x1 - x0, y1 - y0) / grid + 1e-9
    for _ in range(chunks):
        least = np.flatnonzero(visits == visits.min())
        seed_i = least[rng.randint(len(least))]
        cx, cy = xy[seed_i, 0], xy[seed_i, 1]
        pool = _pool_indices(xy, cx, cy, col)
        chosen = (_knn_fallback(xy, rng.randint(n), N) if len(pool) < N // 4
                  else rng.choice(pool, N, replace=len(pool) < N))
        visits[chosen] += 1
        covered[chosen] = True
    return round(100.0 * covered.sum() / n, 3)


def coverage_v5_fps_centers(xy, N, chunks, grid, seed):
    n = len(xy)
    if n <= N:
        return 100.0
    rng = np.random.RandomState(seed)
    k = min(chunks, n)
    idx0 = np.random.RandomState(0).randint(n)   # fixed seed → stable centers (matches training code)
    d2 = ((xy - xy[idx0]) ** 2).sum(1)
    sel = np.empty(k, np.int64)
    sel[0] = idx0
    for i in range(1, k):
        sel[i] = int(d2.argmax())
        d2 = np.minimum(d2, ((xy - xy[sel[i]]) ** 2).sum(1))
    centers = xy[sel]
    col = max(np.ptp(xy[:, 0]), np.ptp(xy[:, 1])) / grid + 1e-9
    covered = np.zeros(n, bool)
    for _ in range(chunks):
        c = centers[rng.randint(len(centers))]
        cx = c[0] + rng.normal(0, col * 0.3)
        cy = c[1] + rng.normal(0, col * 0.3)
        pool = _pool_indices(xy, cx, cy, col)
        chosen = (_knn_fallback(xy, rng.randint(n), N) if len(pool) < N // 4
                  else rng.choice(pool, N, replace=len(pool) < N))
        covered[chosen] = True
    return round(100.0 * covered.sum() / n, 3)


def chunk_coverage_stats(pts_v, cfg):
    '''Runs all 5 variants on the post-voxel points, returns coverage % dict.'''
    xy = pts_v[:, :2]
    N      = cfg["chunk_num_points"]
    chunks = cfg["chunks_per_cloud"]
    grid   = cfg["chunk_grid"]
    seed   = cfg["coverage_seed"]
    return {
        "coverage_pct_v1_randomgrid"     : coverage_v1_random_grid(xy, N, chunks, grid, seed),
        "coverage_pct_v2_sequentialgrid" : coverage_v2_sequential_grid(xy, N, chunks, grid, seed),
        "coverage_pct_v3_slidingwindow"  : coverage_v3_sliding_window(
                                              xy, N, chunks, cfg["window_frac"], cfg["window_overlap"], seed),
        "coverage_pct_v4_coveragebased"  : coverage_v4_coverage_based(xy, N, chunks, grid, seed),
        "coverage_pct_v5_fpscenters"     : coverage_v5_fps_centers(xy, N, chunks, grid, seed),
    }

In [37]:
# ── Master per-file pipeline runner — ties every stage together ──────────────
def process_one_file(path, cfg):
    name = os.path.splitext(os.path.basename(path))[0]
    row = {"filename": name}

    # Stage 1: load
    pts, lbl, load_meta = load_pointcloud(path)
    row.update(load_meta)

    if len(pts) == 0:
        log.warning(f"{name}: 0 valid points after load — skipping remaining stages")
        return row

    # Stage 2: duplicate check (raw, pre-voxel)
    row.update(duplicate_stats(pts))

    # Stage 3: voxel downsample
    voxel_meta, keep_idx = voxel_stage_stats(pts, cfg["voxel_size"])
    row.update(voxel_meta)
    row["voxel_size_used"] = cfg["voxel_size"]

    pts_v = pts[keep_idx]
    lbl_v = lbl[keep_idx] if lbl is not None else None

    # Stage 6a: raw label distribution (before voxel)
    row.update(label_stats(lbl, cfg["target_class"], prefix="raw_"))
    # Stage 6b: post-voxel label distribution (what training actually sees)
    row.update({k: v for k, v in label_stats(lbl_v, cfg["target_class"], prefix="postvoxel_").items()
                if k != "classifications_present"})  # avoid duplicate key

    # Stage 4: normals + height (computed on post-voxel points — same as make_features would see)
    row.update(normal_and_height_stats(pts_v, cfg["normal_radius"], cfg["normal_max_nn"]))
    row["normal_radius_used"]  = cfg["normal_radius"]
    row["normal_max_nn_used"]  = cfg["normal_max_nn"]

    # Stage 5: nn-distance stats (computed on RAW points — scan density, pre-downsample)
    row.update(nn_distance_stats(pts, k=cfg["nn_k"], max_points=cfg["nn_max_points"]))

    # Stage 7: chunk-sampling coverage simulation — all 5 variants (post-voxel points)
    row.update(chunk_coverage_stats(pts_v, cfg))
    row["chunk_num_points_used"] = cfg["chunk_num_points"]
    row["chunks_per_cloud_used"] = cfg["chunks_per_cloud"]

    return row

In [38]:
# ── Run over all files, save CSV ──────────────────────────────────────────────
all_files = list_files(CONFIG["data_dir"])
assert all_files, f"No .las/.laz files found in {CONFIG['data_dir']}"
log.info(f"Found {len(all_files)} files in {CONFIG['data_dir']}")

rows = []
for path in tqdm(all_files, desc="processing files"):
    try:
        rows.append(process_one_file(path, CONFIG))
    except Exception as e:
        log.error(f"{os.path.basename(path)}: FAILED — {e}")
        rows.append({"filename": os.path.splitext(os.path.basename(path))[0],
                    "error": str(e)})

df = pd.DataFrame(rows)
# df.to_csv(CONFIG["output_csv"], index=False)
log.info(f"Saved {len(df)} rows -> {CONFIG['output_csv']}")
df.head()

2026-07-15 00:55:37,109 | INFO | Found 45 files in data/train_v2/


processing files:   0%|          | 0/45 [00:00<?, ?it/s]

2026-07-15 00:57:49,731 | INFO | Saved 45 rows -> outputs/master_pipeline_summary.csv


,filename,xyz_exists,classification_exists,label_field,total_points,nan_count,inf_count,non_finite_dropped,unique_points,duplicate_points,duplicate_percentage,points_after_voxel,voxel_reduction_pct,voxel_size_used,raw_target_points,raw_non_target_points,raw_target_ratio_pct,classifications_present,postvoxel_target_points,postvoxel_non_target_points,postvoxel_target_ratio_pct,centroid_x,centroid_y,centroid_z,scale_norm,x_std,y_std,z_std,height_min,height_max,mean_normal_z,normal_radius_used,normal_max_nn_used,max_nn_distance,mean_nn_distance,median_nn_distance,std_nn_distance,p5_nn_distance,p95_nn_distance,coverage_pct_v1_randomgrid,coverage_pct_v2_sequentialgrid,coverage_pct_v3_slidingwindow,coverage_pct_v4_coveragebased,coverage_pct_v5_fpscenters,chunk_num_points_used,chunks_per_cloud_used
0,sample_data_001,True,True,classification,473438,0,0,0,465623,7815,1.6507,388502,17.9403,0.01,292796,180642,61.8446,"[0, 1]",228062,160440,58.7029,522570.195731,4.520122e+06,75.038960,5.811691,2.113232,1.799308,1.544961,0.0,1.0,0.294525,0.05,30,0.041190,0.008463,0.007626,0.005187,0.000906,0.015027,98.489,97.661,99.338,98.825,96.666,16384,100
1,sample_data_0010,True,True,classification,507790,0,0,0,498902,8888,1.7503,411959,18.8722,0.01,321540,186250,63.3215,"[0, 1]",253431,158528,61.5185,522573.425299,4.520139e+06,79.171380,6.220784,2.331335,2.097774,1.200039,0.0,1.0,0.331508,0.05,30,0.050666,0.007748,0.006800,0.004990,0.000922,0.015027,97.340,97.102,98.617,98.609,97.146,16384,100
2,sample_data_0011,True,True,classification,379774,0,0,0,374453,5321,1.4011,314314,17.2366,0.01,243121,136653,64.0173,"[0, 1]",186707,127607,59.4014,522573.369108,4.520137e+06,77.306296,5.518910,2.005073,2.018249,1.160211,0.0,1.0,0.323974,0.05,30,0.058927,0.008724,0.008103,0.005146,0.000900,0.015001,98.792,98.935,99.462,99.650,99.115,16384,100
3,sample_data_0012,True,True,classification,510604,0,0,0,502015,8589,1.6821,390467,23.5284,0.01,344089,166515,67.3886,"[0, 1]",258839,131628,66.2896,522566.884104,4.520111e+06,71.464434,6.025354,2.152707,1.785417,1.466103,0.0,1.0,0.322518,0.05,30,0.153684,0.006866,0.006083,0.004559,0.000894,0.015000,97.337,97.552,99.526,98.816,97.932,16384,100
4,sample_data_0013,True,True,classification,410573,0,0,0,401510,9063,2.2074,317964,22.5560,0.01,287361,123212,69.9902,"[0, 1]",211939,106025,66.6550,522567.762601,4.520128e+06,78.666119,5.183395,1.942052,1.642549,1.456110,0.0,1.0,0.308413,0.05,30,0.051011,0.007449,0.006650,0.004911,0.000900,0.015027,99.066,99.183,99.303,99.735,99.395,16384,100


In [40]:
df.head(25)

,filename,xyz_exists,classification_exists,label_field,total_points,nan_count,inf_count,non_finite_dropped,unique_points,duplicate_points,duplicate_percentage,points_after_voxel,voxel_reduction_pct,voxel_size_used,raw_target_points,raw_non_target_points,raw_target_ratio_pct,classifications_present,postvoxel_target_points,postvoxel_non_target_points,postvoxel_target_ratio_pct,centroid_x,centroid_y,centroid_z,scale_norm,x_std,y_std,z_std,height_min,height_max,mean_normal_z,normal_radius_used,normal_max_nn_used,max_nn_distance,mean_nn_distance,median_nn_distance,std_nn_distance,p5_nn_distance,p95_nn_distance,coverage_pct_v1_randomgrid,coverage_pct_v2_sequentialgrid,coverage_pct_v3_slidingwindow,coverage_pct_v4_coveragebased,coverage_pct_v5_fpscenters,chunk_num_points_used,chunks_per_cloud_used
0,sample_data_001,True,True,classification,473438,0,0,0,465623,7815,1.6507,388502,17.9403,0.01,292796,180642,61.8446,"[0, 1]",228062,160440,58.7029,522570.195731,4.520122e+06,75.038960,5.811691,2.113232,1.799308,1.544961,0.0,1.0,0.294525,0.05,30,0.041190,0.008463,0.007626,0.005187,0.000906,0.015027,98.489,97.661,99.338,98.825,96.666,16384,100
1,sample_data_0010,True,True,classification,507790,0,0,0,498902,8888,1.7503,411959,18.8722,0.01,321540,186250,63.3215,"[0, 1]",253431,158528,61.5185,522573.425299,4.520139e+06,79.171380,6.220784,2.331335,2.097774,1.200039,0.0,1.0,0.331508,0.05,30,0.050666,0.007748,0.006800,0.004990,0.000922,0.015027,97.340,97.102,98.617,98.609,97.146,16384,100
2,sample_data_0011,True,True,classification,379774,0,0,0,374453,5321,1.4011,314314,17.2366,0.01,243121,136653,64.0173,"[0, 1]",186707,127607,59.4014,522573.369108,4.520137e+06,77.306296,5.518910,2.005073,2.018249,1.160211,0.0,1.0,0.323974,0.05,30,0.058927,0.008724,0.008103,0.005146,0.000900,0.015001,98.792,98.935,99.462,99.650,99.115,16384,100
3,sample_data_0012,True,True,classification,510604,0,0,0,502015,8589,1.6821,390467,23.5284,0.01,344089,166515,67.3886,"[0, 1]",258839,131628,66.2896,522566.884104,4.520111e+06,71.464434,6.025354,2.152707,1.785417,1.466103,0.0,1.0,0.322518,0.05,30,0.153684,0.006866,0.006083,0.004559,0.000894,0.015000,97.337,97.552,99.526,98.816,97.932,16384,100
4,sample_data_0013,True,True,classification,410573,0,0,0,401510,9063,2.2074,317964,22.5560,0.01,287361,123212,69.9902,"[0, 1]",211939,106025,66.6550,522567.762601,4.520128e+06,78.666119,5.183395,1.942052,1.642549,1.456110,0.0,1.0,0.308413,0.05,30,0.051011,0.007449,0.006650,0.004911,0.000900,0.015027,99.066,99.183,99.303,99.735,99.395,16384,100
5,sample_data_0014,True,True,classification,431376,0,0,0,422367,9009,2.0884,325907,24.4494,0.01,306392,124984,71.0267,"[0, 1]",227372,98535,69.7659,522565.463399,4.520118e+06,81.322154,4.941842,1.877937,1.658734,1.446898,0.0,1.0,0.311062,0.05,30,0.028896,0.006335,0.005921,0.004176,0.000922,0.014937,99.193,99.116,99.139,99.690,99.366,16384,100
6,sample_data_0015,True,True,classification,511832,0,0,0,502616,9216,1.8006,396493,22.5345,0.01,365042,146790,71.3207,"[0, 1]",281578,114915,71.0171,522559.471557,4.520128e+06,77.660145,5.488007,2.098379,2.065157,1.070885,0.0,1.0,0.347207,0.05,30,0.117160,0.006954,0.006281,0.004614,0.000900,0.014979,98.027,98.333,98.274,98.933,98.321,16384,100
7,sample_data_0016,True,True,classification,487199,0,0,0,475808,11391,2.3381,388856,20.1854,0.01,331069,156130,67.9535,"[0, 1]",258117,130739,66.3786,522575.732989,4.520127e+06,75.415480,6.426085,1.894409,2.700204,1.424658,0.0,1.0,0.325171,0.05,30,0.036561,0.007760,0.007174,0.005140,0.001100,0.017891,97.060,97.418,92.271,99.075,98.140,16384,100
8,sample_data_0017,True,True,classification,707004,0,0,0,694155,12849,1.8174,567886,19.6771,0.01,509104,197900,72.0086,"[0, 1]",387682,180204,68.2676,522572.222835,4.520131e+06,72.622973,7.358539,2.168108,2.854230,1.368544,0.0,1.0,0.321616,0.05,30,0.035263,0.007832,0.006880,0.005015,0.000900,0.015000,92.225,91.713,92.632,95.256,94.413,16384,100
9,sample_data_0018,True,True,classification,559689,0,0,0,549937,9752,1.7424,453807,1

In [ ]:
df.to_csv("train_data_statistics.csv",index=False)

In [ ]:
# ── Quick sanity view ──────────────────────────────────────────────────────────
pd.set_option("display.max_columns", None)
df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
filename,45,45,sample_data_001,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
xyz_exists,45,1,True,45,NaN,NaN,NaN,NaN,NaN,NaN,NaN
classification_exists,45,1,True,45,NaN,NaN,NaN,NaN,NaN,NaN,NaN
label_field,45,1,classification,45,NaN,NaN,NaN,NaN,NaN,NaN,NaN
total_points,45.0,NaN,NaN,NaN,523146.955556,111001.122925,313288.0,467451.0,507790.0,559387.0,937736.0
nan_count,45.0,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0
inf_count,45.0,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0
non_finite_dropped,45.0,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0
unique_points,45.0,NaN,NaN,NaN,503307.466667,90012.265563,306312.0,461332.0,498902.0,547077.0,730879.0
duplicate_points,45.0,NaN,NaN,NaN,19839.488889,69603.81515,3286.0,8189.0,9752.0,10546.0,476152.0
